# code-review-graph + our tree-sitter module — the 5 repos (no SCIP)

The module is a small tree-sitter walk, defined in this notebook, that fixes the three gaps the language
checks found:
1. **Named functions the graph misses**: JS object methods, `a.b = function`, wrappers like `useCallback`, Java compact constructors and annotation members.
2. **Functions with no name**: each gets an identity = enclosing scope + context label + ordinal:
   `POST /leads handler` · `LeadForm.useEffect#1` · `it('fails')` · `map#2`.
3. **Duplicate names**: identity = `file::scope.name(signature)` — Java parameter types, other languages the parameter count — then `@line` only if two still collide.

Scored against the **same references** as the language checks:

| Repo | Language | Reference |
|---|---|---|
| crm-prototype, crm-prototype-client | JavaScript | TypeScript compiler (`js-ref/functions.mjs`) |
| voice-assessment-agent | Python | `ast` (stdlib) |
| spring-petclinic, guava | Java | javac parser (`java-ref/ListMethods.java`) |

Three scores for **named** functions: code-review-graph · module · both. For **unnamed** functions:
how many of the reference's lambdas/callbacks each one gives a node.

**Self-contained:** nothing from this repo is imported. Only the standard library, `pandas` and
`tree_sitter_language_pack` (the grammar package code-review-graph itself uses). The JS and Java
references are *run* as programs (node / java), not imported. Kernel: **code-graph-rnd (crg 2.4.0)**.

## Parameters

In [1]:
APPS = ['crm-prototype', 'crm-prototype-client', 'voice-assessment-agent', 'spring-petclinic', 'guava']
TINY_LINES = 1   # an unnamed function this many lines or fewer is flagged 'tiny' (e.g. x => x.id); counted, never dropped

## 1. Setup — paths and where each graph is

In [2]:
import ast, datetime as dt, json, os, shutil, sqlite3, subprocess
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd
from tree_sitter_language_pack import get_parser

pd.set_option('display.max_rows', 300); pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 220)

LAB = Path.cwd().resolve().parent if (Path.cwd().resolve().parent / 'apps').exists() else Path.cwd().resolve()
APPS_DIR, RESULTS = LAB / 'apps', LAB / 'results'
JS_REF = LAB / 'js-ref' / 'functions.mjs'
JAVA_REF = LAB / 'java-ref' / 'ListMethods.java'
OUT = RESULTS / 'treesitter-module' / f'{dt.datetime.now():%Y-%m-%d_%H%M}'
OUT.mkdir(parents=True, exist_ok=False)

LANG = {'crm-prototype': 'javascript', 'crm-prototype-client': 'javascript',
        'voice-assessment-agent': 'python', 'spring-petclinic': 'java', 'guava': 'java'}
EXT = {'javascript': ('.js', '.jsx', '.mjs', '.cjs'), 'python': ('.py',), 'java': ('.java',)}
GRAMMAR = {'.js': 'javascript', '.jsx': 'javascript', '.mjs': 'javascript', '.cjs': 'javascript',
           '.ts': 'typescript', '.tsx': 'tsx', '.py': 'python', '.java': 'java'}


def sh(cmd, cwd=None, stdin=None):
    print('$', ' '.join(map(str, cmd))[:300])
    done = subprocess.run([str(c) for c in cmd], cwd=cwd, input=stdin, capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f'{cmd[0]} exited {done.returncode}: {done.stderr[-2000:]}')
    return done.stdout


def latest_graph(app):
    """graph.db of the repo's latest iteration, preferring a code-review-graph 2.4.0 run."""
    log = pd.read_csv(RESULTS / app / 'runs.csv')
    runs = log[log.tool.str.endswith('2.4.0')]
    row = (runs if len(runs) else log).iloc[-1]
    db = RESULTS / app / 'runs' / row.run / 'crg' / 'graph.db'
    if not db.exists():
        raise FileNotFoundError(db)
    return db, row.run, row.tool


graphs = {app: latest_graph(app) for app in APPS}
pd.DataFrame([{'app': a, 'language': LANG[a], 'graph run': r, 'tool': t} for a, (_, r, t) in graphs.items()])

,app,language,graph run,tool
0,crm-prototype,javascript,iter-003_2026-11-18_0743,code-review-graph 2.4.0
1,crm-prototype-client,javascript,iter-006_2026-11-18_0744,code-review-graph 2.4.0
2,voice-assessment-agent,python,iter-005_2026-11-17_2251,code-review-graph 2.4.0
3,spring-petclinic,java,iter-002_2026-11-17_2008,code-review-graph 2.3.8
4,guava,java,iter-003_2026-11-17_2247,code-review-graph 2.4.0


## 2. code-review-graph — function rows from graph.db (read-only)

In [3]:
def crg_functions(app, db):
    root = f'{(APPS_DIR / app).resolve()}/'
    con = sqlite3.connect(f'file:{db}?mode=ro', uri=True)
    files = {p.removeprefix(root) for (p,) in con.execute("select file_path from nodes where kind='File'")}
    df = pd.read_sql_query("select file_path, name, line_start, kind from nodes where kind in ('Function','Test')", con)
    con.close()
    df['file'] = df.file_path.str.removeprefix(root)
    return files, df[['file', 'name', 'line_start', 'kind']]


crg = {}
for app in APPS:
    files, funcs = crg_functions(app, graphs[app][0])
    mine = sorted(f for f in files if f.endswith(EXT[LANG[app]]))
    crg[app] = {'files': mine, 'funcs': funcs[funcs.file.isin(mine)].reset_index(drop=True)}
pd.DataFrame([{'app': a, 'files scored': len(v['files']), 'graph functions': len(v['funcs'])} for a, v in crg.items()])

,app,files scored,graph functions
0,crm-prototype,89,241
1,crm-prototype-client,157,563
2,voice-assessment-agent,72,487
3,spring-petclinic,50,192
4,guava,3267,59188


## 3. Reference — named functions expected, and unnamed ones listed

In [4]:
def ref_python(root, files):
    out = {}
    for rel in files:
        try:
            tree = ast.parse((root / rel).read_bytes())
        except SyntaxError as e:
            out[rel] = {'named': [], 'unnamed': [], 'error': f'Python ast: {e.msg} at line {e.lineno}'}
            continue
        out[rel] = {'named': [(n.name, n.lineno, n.end_lineno) for n in ast.walk(tree)
                              if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))],
                    'unnamed': [(n.lineno, n.end_lineno) for n in ast.walk(tree) if isinstance(n, ast.Lambda)],
                    'error': None}
    return out


def _reshape(raw, label):
    out = {}
    for rel, r in raw.items():
        if 'error' in r:
            out[rel] = {'named': [], 'unnamed': [], 'error': f"{label}: {r['error']}"}
        else:
            out[rel] = {'named': [(f['name'], f['line'], f['end']) for f in r['named']],
                        'unnamed': [(f['line'], f['end']) for f in r['unnamed']],
                        'error': f"{label}: {r['syntax_errors']} syntax error(s)" if r.get('syntax_errors') else None}
    return out


def ref_javascript(root, files):
    return _reshape(json.loads(sh(['node', JS_REF, root, *files])), 'TypeScript compiler')


def jdk_java():
    home = os.environ.get('JAVA_HOME')
    for java in ([str(Path(home) / 'bin' / 'java')] if home else []) + [shutil.which('java') or '']:
        if java and subprocess.run([java, '-version'], capture_output=True).returncode == 0:
            return java
    raise RuntimeError('no working JDK: set JAVA_HOME')


def ref_java(root, files):
    return _reshape(json.loads(sh([jdk_java(), JAVA_REF, root], stdin='\n'.join(files))), 'javac')


REFS = {'python': ref_python, 'javascript': ref_javascript, 'java': ref_java}
expected, could_not_check = {}, []
for app in APPS:
    res = REFS[LANG[app]]((APPS_DIR / app).resolve(), crg[app]['files'])
    expected[app] = res
    could_not_check += [{'app': app, 'file': f, 'reason': r['error']} for f, r in res.items() if r['error']]
could_not_check = pd.DataFrame(could_not_check, columns=['app', 'file', 'reason'])
pd.DataFrame([{'app': a, 'named expected': sum(len(r['named']) for r in e.values()),
               'unnamed listed': sum(len(r['unnamed']) for r in e.values()),
               'files not fully checked': int((could_not_check.app == a).sum())} for a, e in expected.items()])

$ node /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/js-ref/functions.mjs /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/apps/crm-prototype app.js configurations/database.js configurations/env.config.js configurations/logger.config.js configurations/s3.config.js configurations/smtp.config.js conf


$ node /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/js-ref/functions.mjs /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/apps/crm-prototype-client eslint.config.js src/App.jsx src/components/InviteModal.jsx src/components/LanguageToggle.jsx src/components/ProtectedRoute.jsx src/components/app-sid


$ /usr/bin/java /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/java-ref/ListMethods.java /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/apps/spring-petclinic


$ /usr/bin/java /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/java-ref/ListMethods.java /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/apps/guava


,app,named expected,unnamed listed,files not fully checked
0,crm-prototype,227,63,0
1,crm-prototype-client,623,438,0
2,voice-assessment-agent,487,51,0
3,spring-petclinic,195,21,0
4,guava,59236,6153,0


## 4. The module

One walk per file. It keeps a stack of enclosing scopes (class, function, component, test block).

**Named** (the name is in the code): declarations and methods, plus a function *value* that gets a name from where it sits:
`const f = () => {}` · `f = useCallback(() => {})` / `memo(...)` / `forwardRef(...)` · `a.b.c = function(){}` (name `c`) ·
`{ k: () => {} }` (name `k`) · class field `f = () => {}`.

**Unnamed**: everything else. Label from the context:

| Where it sits | Label |
|---|---|
| argument of `router.post('/leads', …)` (get/post/put/patch/delete/all/use/head/options + a string) | `POST /leads handler` |
| argument of `describe('save', …)`, `it('fails', …)`, `test(…)` | `describe('save')`, `it('fails')` |
| argument of a call with a string, e.g. `monkeypatch.setattr(m, "_grade", …)`, `on('data', …)` | `setattr('_grade')`, `on('data')` |
| argument of any other call `useEffect(…)`, `arr.map(…)`, `list.forEach(…)` | `useEffect`, `map`, `forEach` |
| JSX attribute `<Button aria-label="Previous" onClick={() => …}>` | `<Button "Previous">.onClick` (hint: id / aria-label / data-testid / name / title, else the element's text) |
| `return () => …` inside `useEffect` | `cleanup` (elsewhere `returned function`) |
| `x = lambda …` (Python) | `lambda x` |
| `export default () => {}` | `default export` |
| `(function(){})()` | `iife` |
| anywhere else | the node type, e.g. `arrow` |

Then `#n` when the same label repeats in one scope. Keys of nested objects are part of the scope:
`phone: { number: { validate: { validator() {} } } }` → `phone.number.validate.validator`. Identity = `file::scope.path.label#n`.

**Duplicate names**: the graph's own key is `file::scope.name`. The module's key adds the arity
`name(2)`, then `@line` if two functions still share it.

In [5]:
FUNC = {
    'javascript': {'function_declaration', 'generator_function_declaration', 'function_expression', 'function',
                   'generator_function', 'arrow_function', 'method_definition'},
    'python': {'function_definition', 'lambda'},
    'java': {'method_declaration', 'constructor_declaration', 'compact_constructor_declaration',
             'annotation_type_element_declaration', 'lambda_expression'},
}
FUNC['typescript'] = FUNC['tsx'] = FUNC['javascript'] | {'method_signature', 'function_signature', 'abstract_method_signature'}
ALWAYS_NAMED = {'function_declaration', 'generator_function_declaration', 'method_definition', 'method_signature',
                'function_signature', 'abstract_method_signature', 'function_definition', 'method_declaration',
                'constructor_declaration', 'compact_constructor_declaration', 'annotation_type_element_declaration'}
SCOPE = {'class_declaration', 'class', 'abstract_class_declaration', 'interface_declaration', 'enum_declaration',
         'record_declaration', 'annotation_type_declaration', 'class_definition'}
CALLS = {'call_expression', 'method_invocation', 'call', 'new_expression'}
ARGS = {'arguments', 'argument_list'}
WRAPPERS = {'useCallback', 'memo', 'forwardRef'}
ROUTES = {'get', 'post', 'put', 'patch', 'delete', 'all', 'use', 'head', 'options'}
TESTS = {'describe', 'it', 'test', 'context', 'suite', 'specify'}
STRINGS = {'string', 'string_literal', 'template_string'}


def _t(n):
    return n.text.decode('utf-8', 'replace') if n is not None else ''


def _up(n):
    """The parent, skipping parentheses and TS `as` / `satisfies` wrappers."""
    p = n.parent
    while p is not None and p.type in ('parenthesized_expression', 'as_expression', 'satisfies_expression'):
        p = p.parent
    return p


def _callee(call):
    """(name, object text) of a call: `router.post` -> ('post', 'router'), `useEffect` -> ('useEffect', '')."""
    f = call.child_by_field_name('function') or call.child_by_field_name('constructor')
    if call.type == 'method_invocation':
        return _t(call.child_by_field_name('name')), _t(call.child_by_field_name('object'))
    if f is None:
        return '', ''
    if f.type in ('member_expression', 'attribute'):
        prop = f.child_by_field_name('property') or f.child_by_field_name('attribute')
        return _t(prop), _t(f.child_by_field_name('object'))
    return _t(f), ''


def _first_string(args):
    for a in args.named_children:
        if a.type in STRINGS:
            return _t(a).strip('\'"`')
    return None


def _key_text(k):
    return _t(k).strip('\'"`') if k is not None and k.type in ('property_identifier', 'identifier', 'string',
                                                                 'private_property_identifier', 'number') else None


def name_of(n):
    """The function's name if the code gives it one, else None."""
    if n.type in ALWAYS_NAMED:
        return _t(n.child_by_field_name('name')).strip('\'"`') or None
    if n.type in ('lambda', 'lambda_expression'):   # Python / Java lambdas: the references count them as unnamed
        return None
    p = _up(n)
    if p is None:
        return None
    if p.type == 'variable_declarator':
        return _t(p.child_by_field_name('name')) or None
    if p.type in ('assignment_expression', 'augmented_assignment_expression'):
        left = p.child_by_field_name('left')
        if left is not None and left.type == 'member_expression':
            return _key_text(left.child_by_field_name('property'))
        return _t(left) if left is not None and left.type == 'identifier' else None
    if p.type == 'pair':
        return _key_text(p.child_by_field_name('key'))
    if p.type in ('field_definition', 'public_field_definition'):
        return _key_text(p.child_by_field_name('property') or p.child_by_field_name('name'))
    if p.type in ARGS:                                # f = useCallback(() => …), memo(…), forwardRef(…)
        call = _up(p)
        if call is not None and call.type in CALLS and _callee(call)[0] in WRAPPERS:
            q = _up(call)
            if q is not None and q.type == 'variable_declarator':
                return _t(q.child_by_field_name('name')) or None
    return None


JSX_HINTS = ('id', 'aria-label', 'data-testid', 'name', 'title')
EFFECT_HOOKS = ('useEffect', 'useLayoutEffect', 'useInsertionEffect')


def _jsx_label(attr):
    """<Button "Previous">.onClick — element tag, a human hint (id / aria-label / data-testid / name /
    title, else the element's own text), then the attribute."""
    opening = attr.parent                                   # jsx_opening_element / jsx_self_closing_element
    tag = _t(opening.child_by_field_name('name')) or '?'
    hint = None
    for a in opening.named_children:
        if a.type == 'jsx_attribute' and a.named_children and _t(a.named_children[0]) in JSX_HINTS:
            v = a.named_children[-1]
            if v.type == 'string':
                hint = _t(v).strip('\'"'); break
    if hint is None and opening.type == 'jsx_opening_element' and opening.parent is not None:
        texts = [_t(c).strip() for c in opening.parent.named_children if c.type == 'jsx_text' and _t(c).strip()]
        hint = texts[0] if texts else None
    attr_name = _t(attr.named_children[0]) if attr.named_children else 'attr'
    return f'<{tag} "{hint[:40]}">.{attr_name}' if hint else f'<{tag}>.{attr_name}'


def label_of(n, scope=()):
    """A context label for a function with no name."""
    p = _up(n)
    if p is None:
        return n.type
    if p.type == 'jsx_expression' and p.parent is not None and p.parent.type == 'jsx_attribute':
        return _jsx_label(p.parent)
    if p.type in ('return_statement', 'arrow_function') and n.parent is not None:
        # return () => … / useEffect(() => () => …): the cleanup when the enclosing function is an effect
        return 'cleanup' if scope and scope[-1].split('#')[0] in EFFECT_HOOKS else 'returned function'
    if p.type in ARGS:
        call = _up(p)
        if call is not None and call.type in CALLS:
            name, _obj = _callee(call)
            s = _first_string(p)
            if name.lower() in ROUTES and s and s.startswith('/'):
                return f'{name.upper()} {s} handler'
            if s is not None:                         # describe('save'), it('fails'), setattr('_grade'), on('data')
                return f"{name or 'callback'}('{s[:60]}')"
            return name or 'callback'
    if p.type in CALLS and p.child_by_field_name('function') is not None and n.start_byte <= p.child_by_field_name('function').end_byte:
        return 'iife'
    if p.type == 'assignment' and n.type == 'lambda':          # Python: h = lambda x: …
        return f"lambda {_t(p.child_by_field_name('left'))}"
    if p.type == 'export_statement':
        return 'default export'
    if n.type == 'function_expression' and n.child_by_field_name('name') is not None:
        return f"function {_t(n.child_by_field_name('name'))}"
    return {'arrow_function': 'arrow', 'function_expression': 'function', 'lambda': 'lambda',
            'lambda_expression': 'lambda'}.get(n.type, n.type)


def arity(n):
    """The signature part of the identity: Java = parameter types (overloads differ by type), else the count."""
    if n.type in ('method_declaration', 'constructor_declaration'):
        ps = n.child_by_field_name('parameters')
        return ','.join(_t(c.child_by_field_name('type')) or c.type for c in (ps.named_children if ps else [])
                        if c.type in ('formal_parameter', 'spread_parameter'))
    ps = n.child_by_field_name('parameters')
    if ps is None:
        ps = n.child_by_field_name('parameter')   # x => …
        return 1 if ps is not None else 0
    return sum(1 for c in ps.named_children if c.type not in ('comment',))


def module_functions(root, rel):
    """Every function in one file: named and unnamed, each with a scope path and an identity."""
    grammar = GRAMMAR.get(Path(rel).suffix.lower())
    if grammar is None:
        return []
    tree = get_parser(grammar).parse((root / rel).read_bytes())
    funcs, kinds = [], FUNC[grammar]
    counters = defaultdict(Counter)            # scope tuple -> label -> count

    def walk(node, scope):
        new_scope = scope
        if node.is_named and node.type in kinds:
            start, end = node.start_point[0] + 1, node.end_point[0] + 1
            name = name_of(node)
            if name:
                funcs.append({'file': rel, 'name': name, 'line_start': start, 'line_end': end, 'named': True,
                              'scope': '.'.join(scope), 'label': name, 'arity': arity(node)})
                new_scope = scope + (name,)
            else:
                label = label_of(node, scope)
                counters[scope][label] += 1
                label = f'{label}#{counters[scope][label]}'
                funcs.append({'file': rel, 'name': label, 'line_start': start, 'line_end': end, 'named': False,
                              'scope': '.'.join(scope), 'label': label, 'arity': arity(node)})
                new_scope = scope + (label,)
        elif node.type in SCOPE:
            cname = _t(node.child_by_field_name('name'))
            if cname:
                new_scope = scope + (cname,)
        elif node.type == 'pair' and getattr(node.child_by_field_name('value'), 'type', '') == 'object':
            key = _key_text(node.child_by_field_name('key'))
            if key:
                new_scope = scope + (key,)            # phone: { countryCode: { validate: { validator() {} } } }
        elif node.type == 'variable_declarator' and getattr(node.child_by_field_name('value'), 'type', '') == 'object':
            new_scope = scope + (_t(node.child_by_field_name('name')),)   # const api = { save() {} } -> api.save
        for c in node.children:
            walk(c, new_scope)

    walk(tree.root_node, ())
    return funcs


def with_identity(df):
    """identity = file::scope.name(arity), plus @line only where two functions still share it."""
    if df.empty:
        return df.assign(graph_key='', identity='')
    q = df.scope.where(df.scope == '', df.scope + '.') + df.name
    df = df.assign(graph_key=df.file + '::' + q)
    key = df.graph_key + '(' + df.arity.astype(str) + ')'
    clash = key.duplicated(keep=False)
    return df.assign(identity=key.where(~clash, key + '@' + df.line_start.astype(str)))

## 5. Run the module on the scored files

In [6]:
import sys
sys.setrecursionlimit(20000)   # deep trees (guava) walk recursively
mod, mod_errors = {}, []
for app in APPS:
    root, started, rows = (APPS_DIR / app).resolve(), dt.datetime.now(), []
    for rel in crg[app]['files']:
        try:
            rows += module_functions(root, rel)
        except Exception as e:   # listed, never hidden
            mod_errors.append({'app': app, 'file': rel, 'error': repr(e)[:300]})
    df = pd.DataFrame(rows, columns=['file', 'name', 'line_start', 'line_end', 'named', 'scope', 'label', 'arity'])
    df['tiny'] = ~df.named & ((df.line_end - df.line_start + 1) <= TINY_LINES)
    mod[app] = with_identity(df)
    print(f"{app}: {int(df.named.sum())} named, {int((~df.named).sum())} unnamed ({(dt.datetime.now() - started).seconds}s)")
pd.DataFrame(mod_errors, columns=['app', 'file', 'error'])

crm-prototype: 227 named, 63 unnamed (0s)
crm-prototype-client: 623 named, 438 unnamed (0s)
voice-assessment-agent: 487 named, 51 unnamed (0s)
spring-petclinic: 195 named, 21 unnamed (0s)


guava: 59236 named, 6153 unnamed (3s)


,app,file,error


## 6. Named functions — code-review-graph vs module vs both

In [7]:
def match(app, funcs):
    by_file = {f: g for f, g in funcs.groupby('file')}
    empty = funcs.iloc[0:0]
    out = []
    for rel, r in expected[app].items():
        have = by_file.get(rel, empty)
        for name, start, end in r['named']:
            same = have[have.name == name]
            status = ('found' if ((same.line_start >= start) & (same.line_start <= end)).any()
                      else 'lost: same name at another line' if len(same) else 'not found')
            out.append({'app': app, 'file': rel, 'function': name, 'line': start, 'end': end, 'status': status})
    return pd.DataFrame(out, columns=['app', 'file', 'function', 'line', 'end', 'status'])


matched, rows = {}, []
for app in APPS:
    m_named = mod[app][mod[app].named][['file', 'name', 'line_start']]
    for tool, funcs in [('code-review-graph', crg[app]['funcs']), ('module', m_named),
                        ('both', pd.concat([crg[app]['funcs'][['file', 'name', 'line_start']], m_named], ignore_index=True))]:
        m = match(app, funcs); matched[(app, tool)] = m
        found = int((m.status == 'found').sum())
        rows.append({'app': app, 'language': LANG[app], 'tool': tool, 'expected': len(m), 'found': found,
                     'missed': len(m) - found, 'found %': round(100 * found / len(m), 1) if len(m) else None,
                     'lost to same name': int(m.status.str.startswith('lost').sum())})
named_scores = pd.DataFrame(rows)
named_scores.pivot(index=['app', 'language'], columns='tool', values='found %')[['code-review-graph', 'module', 'both']]

,tool,code-review-graph,module,both
app,language,,,
crm-prototype,javascript,90.7,100.0,100.0
crm-prototype-client,javascript,89.7,100.0,100.0
guava,java,99.9,100.0,100.0
spring-petclinic,java,98.5,100.0,100.0
voice-assessment-agent,python,100.0,100.0,100.0


In [8]:
named_scores

,app,language,tool,expected,found,missed,found %,lost to same name
0,crm-prototype,javascript,code-review-graph,227,206,21,90.7,0
1,crm-prototype,javascript,module,227,227,0,100.0,0
2,crm-prototype,javascript,both,227,227,0,100.0,0
3,crm-prototype-client,javascript,code-review-graph,623,559,64,89.7,0
4,crm-prototype-client,javascript,module,623,623,0,100.0,0
5,crm-prototype-client,javascript,both,623,623,0,100.0,0
6,voice-assessment-agent,python,code-review-graph,487,487,0,100.0,0
7,voice-assessment-agent,python,module,487,487,0,100.0,0
8,voice-assessment-agent,python,both,487,487,0,100.0,0
9,spring-petclinic,java,code-review-graph,195,192,3,98.5,3


In [9]:
tot = named_scores.groupby('tool')[['expected', 'found']].sum()
tot['found %'] = (100 * tot.found / tot.expected).round(1)
tot.loc[['code-review-graph', 'module', 'both']]

,expected,found,found %
tool,,,
code-review-graph,60768,60632,99.8
module,60768,60768,100.0
both,60768,60768,100.0


### Who found what (named)

In [10]:
def keys(m):
    return set(map(tuple, m[m.status == 'found'][['app', 'file', 'function', 'line']].values))


who = []
for app in APPS:
    c, s = keys(matched[(app, 'code-review-graph')]), keys(matched[(app, 'module')])
    for rec in matched[(app, 'both')].to_dict('records'):
        k = (rec['app'], rec['file'], rec['function'], rec['line'])
        rec['who'] = ('both' if k in c and k in s else 'code-review-graph only' if k in c
                      else 'module only' if k in s else 'neither')
        who.append(rec)
who = pd.DataFrame(who)
who.groupby(['app', 'who']).size().unstack(fill_value=0)

who,both,module only
app,,
crm-prototype,206,21
crm-prototype-client,559,64
guava,59188,48
spring-petclinic,192,3
voice-assessment-agent,487,0


In [11]:
print('Neither (still missed):'); who[who.who == 'neither'].head(100)

Neither (still missed):


,app,file,function,line,end,status,who


In [12]:
print('Module only (what the module adds):'); who[who.who == 'module only'].head(100)

Module only (what the module adds):


,app,file,function,line,end,status,who
70,crm-prototype,models/companyLead.model.js,validator,26,29,found,module only
71,crm-prototype,models/companyLead.model.js,validator,38,41,found,module only
72,crm-prototype,models/companyLead.model.js,validator,49,52,found,module only
75,crm-prototype,models/companyProspectus.model.js,validator,26,29,found,module only
76,crm-prototype,models/companyProspectus.model.js,validator,39,42,found,module only
77,crm-prototype,models/companyProspectus.model.js,validator,50,53,found,module only
80,crm-prototype,models/contactLead.model.js,validator,17,20,found,module only
81,crm-prototype,models/contactLead.model.js,validator,29,32,found,module only
82,crm-prototype,models/contactLead.model.js,validator,40,43,found,module only
85,crm-prototype,models/contactProspectus.model.js,validator,16,19,found,module only


In [13]:
print('code-review-graph only (what the module misses — rules to add):'); who[who.who == 'code-review-graph only'].head(100)

code-review-graph only (what the module misses — rules to add):


,app,file,function,line,end,status,who


## 7. Unnamed functions — does each reference lambda/callback get a node?
Counted per file and start line: covered = min(reference unnamed functions starting there, the tool's unnamed rows starting there).
For code-review-graph (no unnamed flag) its rows on that line minus the reference's named functions there. code-review-graph names only test blocks (`it@L45`), so it is
expected to be low.

In [14]:
def unnamed_cover(app, unnamed_lines_by_file):
    """Per file and start line: covered = min(reference unnamed there, tool's unnamed rows there).
    Counted per line, so a callback on the same line as a named function still counts."""
    total = hit = 0
    for rel, r in expected[app].items():
        need = Counter(s for s, _ in r['unnamed'])
        have = unnamed_lines_by_file.get(rel, Counter())
        total += sum(need.values())
        hit += sum(min(k, have[line]) for line, k in need.items())
    return total, hit


rows = []
for app in APPS:
    # code-review-graph has no unnamed flag: its rows on a line, minus the reference's named functions there
    c = {}
    for f, g in crg[app]['funcs'].groupby('file'):
        rows_at = Counter(g.line_start)
        named_at = Counter(s for _, s, _ in expected[app].get(f, {'named': []})['named'])
        c[f] = Counter({line: max(0, k - named_at[line]) for line, k in rows_at.items()})
    mu = mod[app][~mod[app].named]
    m = {f: Counter(g.line_start) for f, g in mu.groupby('file')}
    for tool, starts in [('code-review-graph', c), ('module', m)]:
        total, hit = unnamed_cover(app, starts)
        rows.append({'app': app, 'tool': tool, 'reference unnamed': total, 'given a node': hit,
                     '%': round(100 * hit / total, 1) if total else None})
    rows.append({'app': app, 'tool': 'module (of which tiny)', 'reference unnamed': None,
                 'given a node': int(mu.tiny.sum()), '%': None})
unnamed_scores = pd.DataFrame(rows)
unnamed_scores

,app,tool,reference unnamed,given a node,%
0,crm-prototype,code-review-graph,63.0,35,55.6
1,crm-prototype,module,63.0,63,100.0
2,crm-prototype,module (of which tiny),NaN,17,NaN
3,crm-prototype-client,code-review-graph,438.0,3,0.7
4,crm-prototype-client,module,438.0,438,100.0
5,crm-prototype-client,module (of which tiny),NaN,161,NaN
6,voice-assessment-agent,code-review-graph,51.0,0,0.0
7,voice-assessment-agent,module,51.0,51,100.0
8,voice-assessment-agent,module (of which tiny),NaN,46,NaN
9,spring-petclinic,code-review-graph,21.0,0,0.0


### The labels the module gave — for review

In [15]:
allu = pd.concat([mod[a][~mod[a].named].assign(app=a) for a in APPS], ignore_index=True)
allu['label kind'] = allu.label.str.replace(r'#\d+$', '', regex=True).str.replace(r"\('.*'\)$", "('…')", regex=True) \
                                .str.replace(r'^(GET|POST|PUT|PATCH|DELETE|ALL|USE|HEAD|OPTIONS) .* handler$', r'\1 <path> handler', regex=True)
allu.groupby(['app', 'label kind']).size().sort_values(ascending=False).head(40)

app                     label kind         
guava                   assertThrows           4445
                        lambda                  424
                        of                      142
                        transform                90
                        returned function        77
                        forEach                  75
                        submit                   68
crm-prototype-client    map                      67
guava                   execute                  48
                        addListener              46
crm-prototype-client    useEffect                43
                        <Button>.onClick         37
voice-assessment-agent  setattr('…')             36
guava                   replaceAll               33
                        call                     30
                        compute                  30
                        computeIfPresent         25
crm-prototype-client    useUserStore             25
                    

In [16]:
cols = ['app', 'file', 'line_start', 'line_end', 'scope', 'label', 'identity', 'tiny']
allu[~allu.tiny].groupby('app').head(15)[cols]

,app,file,line_start,line_end,scope,label,identity,tiny
0,crm-prototype,app.js,36,38,,GET / handler#1,app.js::GET / handler#1(2),False
1,crm-prototype,configurations/env.config.js,15,20,,forEach#1,configurations/env.config.js::forEach#1(1),False
3,crm-prototype,middlewares/auth.middleware.js,12,16,parseCookies,reduce#1,middlewares/auth.middleware.js::parseCookies.reduce#1(2),False
5,crm-prototype,middlewares/auth.middleware.js,121,149,allowRoles,returned function#1,middlewares/auth.middleware.js::allowRoles.returned function#1(3),False
6,crm-prototype,models/callLog.model.js,77,79,createCallLogSchema,pre#1,models/callLog.model.js::createCallLogSchema.pre#1(0),False
7,crm-prototype,models/companyLead.model.js,178,180,createCompanyLeadSchema,pre#1,models/companyLead.model.js::createCompanyLeadSchema.pre#1(0),False
8,crm-prototype,models/companyProspectus.model.js,174,176,createCompanyProspectusSchema,pre#1,models/companyProspectus.model.js::createCompanyProspectusSchema.pre#1(0),False
9,crm-prototype,models/contactLead.model.js,98,100,createContactLeadSchema,pre#1,models/contactLead.model.js::createContactLeadSchema.pre#1(0),False
10,crm-prototype,models/contactProspectus.model.js,97,99,createContactProspectusSchema,pre#1,models/contactProspectus.model.js::createContactProspectusSchema.pre#1(0),False
11,crm-prototype,models/lead.model.js,138,140,createLeadSchema,pre#1,models/lead.model.js::createLeadSchema.pre#1(0),False


## 8. Duplicate names — what the graph's key collapses, and what the module's identity keeps apart
`graph key` = `file::scope.name` (how code-review-graph keys a node). A group of 2+ named functions with
the same graph key is stored as **one** node in the graph. The module's identity adds `(arity)`, then `@line`.

In [17]:
rows, examples = [], []
for app in APPS:
    n = mod[app][mod[app].named]
    g = n.groupby('graph_key').size()
    dup_keys = g[g > 1]
    by_arity = n.assign(k=n.graph_key + '(' + n.arity.astype(str) + ')').groupby('k').size()
    rows.append({'app': app, 'named functions': len(n),
                 'graph-key groups with 2+': len(dup_keys), 'functions in them': int(dup_keys.sum()),
                 'still clash after (arity)': int(by_arity[by_arity > 1].sum()),
                 'unique after identity': int(n.identity.is_unique)})
    examples.append(n[n.graph_key.isin(dup_keys.index)].assign(app=app).head(12))
dupes = pd.DataFrame(rows)
dupes

,app,named functions,graph-key groups with 2+,functions in them,still clash after (arity),unique after identity
0,crm-prototype,227,0,0,0,1
1,crm-prototype-client,623,0,0,0,1
2,voice-assessment-agent,487,0,0,0,1
3,spring-petclinic,195,2,5,0,1
4,guava,59236,2827,7762,1723,1


In [18]:
pd.concat(examples)[['app', 'graph_key', 'arity', 'line_start', 'identity']] if examples else None

,app,graph_key,arity,line_start,identity
23,spring-petclinic,src/main/java/org/springframework/samples/petclinic/owner/Owner.java::Owner.getPet,String,120,src/main/java/org/springframework/samples/petclinic/owner/Owner.java::Owner.getPet(String)
24,spring-petclinic,src/main/java/org/springframework/samples/petclinic/owner/Owner.java::Owner.getPet,Integer,129,src/main/java/org/springframework/samples/petclinic/owner/Owner.java::Owner.getPet(Integer)
25,spring-petclinic,src/main/java/org/springframework/samples/petclinic/owner/Owner.java::Owner.getPet,"String,boolean",147,"src/main/java/org/springframework/samples/petclinic/owner/Owner.java::Owner.getPet(String,boolean)"
99,spring-petclinic,src/main/java/org/springframework/samples/petclinic/vet/VetRepository.java::VetRepository.findAll,,44,src/main/java/org/springframework/samples/petclinic/vet/VetRepository.java::VetRepository.findAll()
100,spring-petclinic,src/main/java/org/springframework/samples/petclinic/vet/VetRepository.java::VetRepository.findAll,Pageable,54,src/main/java/org/springframework/samples/petclinic/vet/VetRepository.java::VetRepository.findAll(Pageable)
9,guava,android/guava-testlib/src/com/google/common/collect/testing/AbstractContainerTester.java::AbstractContainerTester.re...,,69,android/guava-testlib/src/com/google/common/collect/testing/AbstractContainerTester.java::AbstractContainerTester.re...
10,guava,android/guava-testlib/src/com/google/common/collect/testing/AbstractContainerTester.java::AbstractContainerTester.re...,C,84,android/guava-testlib/src/com/google/common/collect/testing/AbstractContainerTester.java::AbstractContainerTester.re...
11,guava,android/guava-testlib/src/com/google/common/collect/testing/AbstractContainerTester.java::AbstractContainerTester.ex...,spread_parameter,94,android/guava-testlib/src/com/google/common/collect/testing/AbstractContainerTester.java::AbstractContainerTester.ex...
12,guava,android/guava-testlib/src/com/google/common/collect/testing/AbstractContainerTester.java::AbstractContainerTester.ex...,Collection<E>,114,android/guava-testlib/src/com/google/common/collect/testing/AbstractContainerTester.java::AbstractContainerTester.ex...
14,guava,android/guava-testlib/src/com/google/common/collect/testing/AbstractContainerTester.java::AbstractContainerTester.ex...,spread_parameter,141,android/guava-testlib/src/com/google/common/collect/testing/AbstractContainerTester.java::AbstractContainerTester.ex...


## 9. Save

In [19]:
named_scores.to_csv(OUT / 'named_scores.csv', index=False)
unnamed_scores.to_csv(OUT / 'unnamed_scores.csv', index=False)
dupes.to_csv(OUT / 'duplicates.csv', index=False)
who.to_csv(OUT / 'who_found_what.csv', index=False)
pd.concat([mod[a].assign(app=a) for a in APPS], ignore_index=True).to_csv(OUT / 'module_functions.csv', index=False)
could_not_check.to_csv(OUT / 'could_not_check.csv', index=False)
pd.DataFrame(mod_errors, columns=['app', 'file', 'error']).to_csv(OUT / 'module_errors.csv', index=False)
(OUT / 'run.json').write_text(json.dumps({'apps': APPS, 'graphs': {a: [str(d), r, t] for a, (d, r, t) in graphs.items()},
                                          'run_at': dt.datetime.now().isoformat(timespec='seconds')}, indent=2))
sorted(p.name for p in OUT.iterdir())

['could_not_check.csv',
 'duplicates.csv',
 'module_errors.csv',
 'module_functions.csv',
 'named_scores.csv',
 'run.json',
 'unnamed_scores.csv',
 'who_found_what.csv']